# AI VIET NAM – COURSE 2023
## Exercise: Transformer Applications
**Mục tiêu:**
1. Cài đặt chi tiết kiến trúc Transformer từ đầu bằng PyTorch (Encoder, Decoder, Seq2Seq).
2. Ứng dụng Transformer-Encoder vào bài toán Text Classification (Dataset: NTC-SCV).
3. Fine-tuning mô hình Pretrained BERT (`bert-base-uncased`) cho bài toán Text Classification.
4. Xây dựng Vision Transformer (ViT) từ đầu và Fine-tuning Pretrained ViT (`google/vit-base-patch16-224-in21k`) trên tập dữ liệu hoa (`flower_photos`).
5. Giải đáp và phân tích chi tiết 10 câu hỏi trắc nghiệm ôn tập lý thuyết.


---
## 0. Import Thư Viện & Cấu Hình Môi Trường


In [1]:
import os
import random
import re
import string
import time
import math
from typing import List, Tuple, Dict, Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

# Import các components từ module transformer.py
from transformer import (
    set_seed,
    preprocess_text,
    TokenAndPositionEmbedding,
    TransformerEncoderBlock,
    TransformerEncoder,
    TransformerDecoderBlock,
    TransformerDecoder,
    Transformer,
    TransformerEncoderCls,
    PatchPositionEmbedding,
    VisionTransformerCls,
    train_epoch,
    evaluate_epoch,
    train_model,
    plot_result,
)

# Thiết lập seed đảm bảo tính tái lập kết quả
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device đang sử dụng: {device}")


Device đang sử dụng: cuda


---
## Phần 1. Kiến Trúc Transformer (Xây Dựng Từ Đầu Bằng PyTorch)

Kiến trúc Transformer theo bài báo *"Attention Is All You Need"* gồm 2 thành phần chính:
- **Encoder:** Mã hoá các tokens đầu vào thành contextual embeddings (Multi-Head Attention, Add & LayerNorm, Feed-Forward).
- **Decoder:** Nhận token lịch sử và trạng thái từ Encoder để sinh token tiếp theo (Masked Multi-Head Attention, Cross-Attention, Add & LayerNorm, Feed-Forward).
- **Causal Masking:** Đảm bảo quá trình sinh từ không nhìn thấy trước các token trong tương lai.


In [2]:
# Thử nghiệm khởi tạo mô hình Transformer hoàn chỉnh
batch_size = 128
src_vocab_size = 1000
tgt_vocab_size = 2000
embed_dim = 200
max_length = 100
num_layers = 2
num_heads = 4
ff_dim = 256

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    embed_dim=embed_dim,
    max_length=max_length,
    num_layers=num_layers,
    num_heads=num_heads,
    ff_dim=ff_dim,
    device=device,
).to(device)

print(model)


Transformer(
  (encoder): TransformerEncoder(
    (embedding): TokenAndPositionEmbedding(
      (word_emb): Embedding(1000, 200)
      (pos_emb): Embedding(100, 200)
    )
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
        )
        (ffn): Sequential(
          (0): Linear(in_features=200, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=200, bias=True)
        )
        (layernorm_1): LayerNorm((200,), eps=1e-06, elementwise_affine=True)
        (layernorm_2): LayerNorm((200,), eps=1e-06, elementwise_affine=True)
        (dropout_1): Dropout(p=0.1, inplace=False)
        (dropout_2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): TransformerDecoder(
    (embedding): TokenAndPositionEmbedding(
      (word_emb): Embedding(2000, 200)
      (pos_emb): Embedding(10

/mnt/data/Code/ai/llm/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Ti with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [ ]:
# Kiểm tra kích thước tensor đầu vào và đầu ra của mô hình Transformer
src = torch.randint(
    high=src_vocab_size,
    size=(batch_size, max_length),
    dtype=torch.int64,
    device=device,
)
tgt = torch.randint(
    high=tgt_vocab_size,
    size=(batch_size, max_length),
    dtype=torch.int64,
    device=device,
)

with torch.no_grad():
    prediction = model(src, tgt)

print(f"Source shape:      {src.shape} (batch_size, max_length)")
print(f"Target shape:      {tgt.shape} (batch_size, max_length)")
print(f"Prediction shape:  {prediction.shape} (batch_size, max_length, tgt_vocab_size)")
assert prediction.shape == (batch_size, max_length, tgt_vocab_size), "Shape không khớp!"
print("=> Khởi tạo và kiểm thử Transformer Seq2Seq thành công!")


---
## Phần 2. Text Classification Sử Dụng Transformer-Encoder

Trong phần này, ta sử dụng phần **Encoder** của Transformer kết hợp với **Average Pooling** và **MLP Classifier** để phân loại cảm xúc văn bản trên bộ dữ liệu `NTC-SCV`.


In [ ]:
# 1. Tải bộ dữ liệu NTC-SCV
!git clone https://github.com/congnghia0609/ntc-scv.git
!mkdir -p ./data
!unzip -q -o ./ntc-scv/data/data_test.zip -d ./data
!unzip -q -o ./ntc-scv/data/data_train.zip -d ./data
!rm -rf ./ntc-scv
print("Đã tải và giải nén dữ liệu thành công!")


In [ ]:
# 2. Đọc dữ liệu vào DataFrame
import pandas as pd

def load_data_from_path(folder_path: str) -> pd.DataFrame:
    examples = []
    for label_name in os.listdir(folder_path):
        label_dir = os.path.join(folder_path, label_name)
        if not os.path.isdir(label_dir):
            continue
        label = 0 if label_name == "neg" else 1 if label_name == "pos" else None
        if label is None:
            continue
        for file_name in os.listdir(label_dir):
            file_path = os.path.join(label_dir, file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            sentence = " ".join(lines).strip()
            examples.append({"sentence": sentence, "label": label})
    return pd.DataFrame(examples)

train_df = load_data_from_path("./data/data_train/train")
valid_df = load_data_from_path("./data/data_train/test")
test_df = load_data_from_path("./data/data_test/test")

print(f"Kích thước tập Train: {len(train_df)}")
print(f"Kích thước tập Valid: {len(valid_df)}")
print(f"Kích thước tập Test:  {len(test_df)}")


In [ ]:
# 3. Tiền xử lý dữ liệu text
train_df["preprocess_sentence"] = train_df["sentence"].apply(preprocess_text)
valid_df["preprocess_sentence"] = valid_df["sentence"].apply(preprocess_text)
test_df["preprocess_sentence"] = test_df["sentence"].apply(preprocess_text)

print("Ví dụ sau tiền xử lý:")
print("Gốc:       ", train_df["sentence"].iloc[0][:100])
print("Tiền xử lý:", train_df["preprocess_sentence"].iloc[0][:100])


In [ ]:
# 4. Tokenization & Xây dựng Vocabulary (dùng torchtext hoặc tokenizer cơ bản)
try:
    from torchtext.data.utils import get_tokenizer
    from torchtext.vocab import build_vocab_from_iterator
    from torchtext.data.functional import to_map_style_dataset

    tokenizer = get_tokenizer("basic_english")
    def yield_tokens(sentences, tok):
        for s in sentences:
            yield tok(s)

    vocab_size = 10000
    vocabulary = build_vocab_from_iterator(
        yield_tokens(train_df["preprocess_sentence"], tokenizer),
        max_tokens=vocab_size,
        specials=["<pad>", "<unk>"]
    )
    vocabulary.set_default_index(vocabulary["<unk>"])

    def prepare_dataset(df):
        for _, row in df.iterrows():
            encoded = vocabulary(tokenizer(row["preprocess_sentence"]))
            yield encoded, row["label"]

    train_dataset = to_map_style_dataset(prepare_dataset(train_df))
    valid_dataset = to_map_style_dataset(prepare_dataset(valid_df))
    test_dataset = to_map_style_dataset(prepare_dataset(test_df))
    print("Xây dựng Vocabulary thành công với torchtext!")
except ImportError:
    print("torchtext không khả dụng, sử dụng custom tokenizer đơn giản...")
    from collections import Counter

    tokenizer = lambda s: s.split()
    counter = Counter()
    for s in train_df["preprocess_sentence"]:
        counter.update(tokenizer(s))
    
    vocab_words = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(9998)]
    word2idx = {w: i for i, w in enumerate(vocab_words)}
    unk_idx = word2idx["<unk>"]

    def encode(s):
        return [word2idx.get(w, unk_idx) for w in tokenizer(s)]

    train_dataset = [(encode(r["preprocess_sentence"]), r["label"]) for _, r in train_df.iterrows()]
    valid_dataset = [(encode(r["preprocess_sentence"]), r["label"]) for _, r in valid_df.iterrows()]
    test_dataset = [(encode(r["preprocess_sentence"]), r["label"]) for _, r in test_df.iterrows()]
    vocab_size = len(word2idx)
    print(f"Custom vocabulary size: {vocab_size}")


In [ ]:
# 5. Dataloader & Padding Collate Function
seq_length = 100

def collate_batch(batch):
    sentences, labels = list(zip(*batch))
    encoded_sentences = [
        s + [0] * (seq_length - len(s)) if len(s) < seq_length else s[:seq_length]
        for s in sentences
    ]
    encoded_sentences = torch.tensor(encoded_sentences, dtype=torch.int64)
    labels = torch.tensor(labels, dtype=torch.int64)
    return encoded_sentences, labels

batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
print("DataLoaders đã sẵn sàng!")


In [ ]:
# 6. Khởi tạo mô hình TransformerEncoderCls và Huấn luyện
model_cls = TransformerEncoderCls(
    vocab_size=10000,
    max_length=100,
    num_layers=2,
    embed_dim=200,
    num_heads=4,
    ff_dim=128,
    dropout=0.1,
    device=device,
    num_classes=2,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cls.parameters(), lr=0.00005)

# Huấn luyện 50 epochs (có thể giảm xuống để chạy thử nhanh)
num_epochs = 50
trained_model, metrics = train_model(
    model=model_cls,
    model_name="transformer_encoder_cls",
    save_model_dir="./saved_models",
    optimizer=optimizer,
    criterion=criterion,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    num_epochs=num_epochs,
    device=device,
)

# Đánh giá trên tập test
test_acc, test_loss = evaluate_epoch(trained_model, criterion, test_dataloader, device)
print(f"\n=> Độ chính xác trên tập Test: {test_acc * 100:.2f}% (Trong tài liệu đạt khoảng 82.55%)")

# Vẽ biểu đồ Accuracy và Loss
plot_result(num_epochs, metrics["train_accuracy"], metrics["valid_accuracy"], metrics["train_loss"], metrics["valid_loss"])


---
## Phần 3. Text Classification Sử Dụng Pretrained BERT

Trong phần này, ta fine-tune mô hình `bert-base-uncased` cho bài toán phân loại văn bản `NTC-SCV` bằng thư viện `transformers` của Hugging Face.


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q -U transformers datasets accelerate evaluate


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

# Chuyển đổi pandas DataFrames thành Hugging Face DatasetDict
raw_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "valid": Dataset.from_pandas(valid_df),
    "test": Dataset.from_pandas(test_df),
})

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

def preprocess_function(examples):
    result = tokenizer(
        examples["preprocess_sentence"],
        padding="max_length",
        max_length=100,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

processed_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    desc="Running tokenizer on dataset",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# Cấu hình mô hình BERT và hàm tính Accuracy
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    finetuning_task="text-classification",
)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config,
)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./saved_models/bert_model",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["valid"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Chạy huấn luyện BERT
trainer.train()

# Đánh giá kết quả trên tập Test
test_results = trainer.predict(processed_dataset["test"])
print(f"\n=> Độ chính xác Fine-tuning BERT trên tập Test: {test_results.metrics.get('test_accuracy', 0)*100:.2f}% (Tài liệu đạt ~85.52%)")


---
## Phần 4. Vision Transformer (ViT)

Vision Transformer (ViT) áp dụng cơ chế Self-Attention trực tiếp lên các mảnh ảnh (Patches).
- Ảnh kích thước $224 \times 224$ được chia thành các patch $16 \times 16$.
- Mỗi patch được chiếu tuyến tính qua `nn.Conv2d` thành vector embedding kích thước `embed_dim=512`.
- Cộng thêm vector vị trí học được (`positional_embedding`).
- Đưa qua các khối `TransformerEncoderBlock` để trích xuất đặc trưng.


In [ ]:
# 1. Tải bộ dữ liệu Flower Photos
!gdown 11Buzytn4vIh4x_Oqz8MY29JMMdIqSzj-
!unzip -q -o ./flower_photos.zip
print("Đã tải và giải nén flower_photos!")


In [ ]:
# 2. Data Augmentation & DataLoader cho Ảnh
from torchvision import transforms
from torchvision.datasets import ImageFolder

IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = ImageFolder(root="./flower_photos")
num_samples = len(dataset)
classes = dataset.classes
num_classes = len(classes)
print(f"Số lượng mẫu: {num_samples}, Số lớp ({num_classes}): {classes}")

TRAIN_RATIO, VALID_RATIO = 0.8, 0.1
n_train = int(num_samples * TRAIN_RATIO)
n_valid = int(num_samples * VALID_RATIO)
n_test = num_samples - n_train - n_valid

train_dataset, valid_dataset, test_dataset = random_split(dataset, [n_train, n_valid, n_test])
train_dataset.dataset.transform = train_transforms
valid_dataset.dataset.transform = test_transforms
test_dataset.dataset.transform = test_transforms

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [ ]:
# 3. Huấn luyện Vision Transformer từ đầu (ViT Scratch)
vit_scratch = VisionTransformerCls(
    image_size=224,
    embed_dim=512,
    num_heads=4,
    ff_dim=128,
    dropout=0.1,
    device=device,
    num_classes=num_classes,
    patch_size=16,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vit_scratch.parameters(), lr=0.00005)

# Huấn luyện 50 epochs (kết quả trong tài liệu đạt ~63.49%)
trained_vit, vit_metrics = train_model(
    model=vit_scratch,
    model_name="vit_flowers_scratch",
    save_model_dir="./saved_models/vit_scratch",
    optimizer=optimizer,
    criterion=criterion,
    train_dataloader=train_loader,
    valid_dataloader=val_loader,
    num_epochs=50,
    device=device,
)

test_acc, test_loss = evaluate_epoch(trained_vit, criterion, test_loader, device)
print(f"\n=> Độ chính xác ViT Scratch trên tập Test: {test_acc * 100:.2f}% (Tài liệu đạt ~63.49%)")


In [ ]:
# 4. Fine-Tuning Pretrained Vision Transformer (ViT)
from transformers import ViTForImageClassification, ViTImageProcessor

model_checkpoint = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTImageProcessor.from_pretrained(model_checkpoint)

id2label = {i: label for i, label in enumerate(classes)}
label2id = {label: i for i, label in enumerate(classes)}

pretrained_vit = ViTForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
).to(device)

def collate_fn_vit(examples):
    pixel_values = torch.stack([example[0] for example in examples])
    labels = torch.tensor([example[1] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

vit_args = TrainingArguments(
    output_dir="./saved_models/vit_flowers_pretrained",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
)

vit_trainer = Trainer(
    model=pretrained_vit,
    args=vit_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collate_fn_vit,
    compute_metrics=compute_metrics,
    processing_class=feature_extractor,
)

vit_trainer.train()
vit_test_output = vit_trainer.predict(test_dataset)
print(f"\n=> Độ chính xác Pretrained ViT trên tập Test: {vit_test_output.metrics.get('test_accuracy', 0)*100:.2f}% (Tài liệu đạt ~95.1%)")


---
## Phần 5. Đáp Án & Giải Thích Chi Tiết 10 Câu Hỏi Trắc Nghiệm (Phần 4 PDF)

| Câu hỏi | Câu hỏi & Lựa chọn | Đáp án đúng | Giải thích chi tiết |
|:---:|:---|:---:|:---|
| **Câu 1** | **Lớp Positional Encoding trong Transformer dùng để làm gì?**<br>a) Biểu diễn vị trí của các tokens<br>b) Tính Attention<br>c) Dự đoán token tiếp theo<br>d) Tính Loss | **a** | Cơ chế Self-Attention không có tính tuần tự (order-invariant). Do đó, Positional Encoding cần được cộng vào Word Embedding để cung cấp thông tin về thứ tự xuất hiện của từng từ trong câu. |
| **Câu 2** | **Layer nào sau đây không có trong Transformer-Encoder?**<br>a) Masked Multi-Head Attention<br>b) Multi-Head Attention<br>c) Layer Normalization<br>d) Feed Forward | **a** | `Masked Multi-Head Attention` chỉ xuất hiện ở **Decoder** nhằm che các từ phía sau khi tự sinh từ. Encoder xử lý toàn bộ câu hai chiều nên dùng `Multi-Head Attention` thông thường. |
| **Câu 3** | **Layer nào sau đây không có trong Transformer-Decoder?**<br>a) Masked Multi-Head Attention<br>b) Multi-Head Attention<br>c) Positional Encoding<br>d) Feed Forward | **c** | `Positional Encoding` là lớp embedding độc lập ở bước tiền xử lý input/output embedding trước khi đưa vào các khối Encoder/Decoder, không phải là một sub-layer nằm bên trong `Transformer-DecoderBlock`. |
| **Câu 4** | **Masked Multi-Head Attention được sử dụng để làm gì?**<br>a) Mask những token lịch sử<br>b) Mask vị trí các token lịch sử<br>c) Mask những token trong tương lai<br>d) Mask cả những token lịch sử và token trong tương lai | **c** | Trong quá trình giải mã tuần tự (autoregressive decoding), tại bước $t$, mô hình không được phép nhìn thấy các token từ bước $t+1$ trở đi. Do đó cần causal mask để che (mask) các token trong tương lai. |
| **Câu 5** | **Phát biểu nào sau đây là đúng về BERT?**<br>a) Bao gồm các khối Transformer-Encoder<br>b) Bao gồm các khối Transformer-Decoder<br>c) Cả 2 đáp án a và b đều đúng<br>d) Cả 2 đáp án a và b đều sai | **a** | BERT (Bidirectional Encoder Representations from Transformers) được xây dựng hoàn toàn từ chồng các khối **Transformer-Encoder** hai chiều. |
| **Câu 6** | **Objective Function của BERT là?**<br>a) Masked Language Model (MLM)<br>b) Next Sentence Prediction (NSP)<br>c) Cả a và b đều đúng<br>d) Cả a và b đều sai | **c** | BERT được huấn luyện trước (pre-trained) đồng thời với 2 mục tiêu: **MLM** (dự đoán token bị ẩn ngẫu nhiên 15%) và **NSP** (dự đoán câu B có nối tiếp câu A không). |
| **Câu 7** | **Thành phần nào không có trong các giá trị Input của BERT?**<br>a) Token Embeddings<br>b) Segment Embeddings<br>c) Position Embeddings<br>d) Language Model Head | **d** | Input của BERT gồm tổng của 3 embedding: Token + Segment + Position. Còn `Language Model Head` là tầng đầu ra (output head), không phải giá trị đầu vào. |
| **Câu 8** | **Số lượng tham số của mô hình BERT-base và BERT-large là?**<br>a) 340M và 110M<br>b) 110M và 110M<br>c) 340M và 340M<br>d) 110M và 340M | **d** | Theo bài báo nguyên bản BERT (Devlin et al., 2018): **BERT-base** có 110 triệu tham số (12 layers, hidden 768), còn **BERT-large** có 340 triệu tham số (24 layers, hidden 1024). |
| **Câu 9** | **Các Patch trong Vision Transformer được xác định như thế nào?**<br>a) Flatten ảnh đầu vào<br>b) Trung bình các giá trị điểm ảnh theo hàng<br>c) Trung bình các giá trị điểm ảnh theo cột<br>d) Chia ảnh đầu vào thành các Patch với kích thước các mảng cố định | **d** | ViT chia ảnh gốc $H \times W$ thành một chuỗi các patch không chồng lấn có kích thước cố định $P \times P$ (ví dụ: $16 \times 16$) rồi làm phẳng và chiếu thành vector embedding. |
| **Câu 10** | **Bộ dữ liệu nào sau đây được sử dụng trong phần thực nghiệm so sánh kết quả Vision Transformer với các phương pháp huấn luyện từ đầu và sử dụng pretrained?**<br>a) CIFAR10<br>b) Flower<br>c) CIFAR100<br>d) MNIST | **b** | Trong tài liệu PDF (Phần 3, trang 20 - 24), thực nghiệm so sánh ViT Scratch (63.49%) và Pretrained ViT (95.1%) được thực hiện trên bộ dữ liệu `flower_photos` (Flower). |
